In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Cargar datos de la Capa Bronze
df_bronze = spark.table("telemetria_patio_bronze")

# 2. PIVOT: Convertir métricas en columnas reales
# Agrupamos por timestamp (ts) para reconstruir cada evento
df_pivoted = df_bronze.groupBy("ts").pivot("metrica").agg(F.first("valor"))

# 3. LIMPIEZA Y CASTING (Reglas de la Capa Silver)
df_silver = df_pivoted.select(
    F.col("ts").cast("long").alias("evento_ts"),
    F.col("logs_nia").cast("long"),
    F.col("logs_ubicacion"),
    F.col("shared_tipo"),
    F.col("shared_conductor"),
    F.col("shared_empresa"),
    F.col("shared_placaTracto"),
    F.col("shared_placaPlataforma")
)

# 4. APLICAR FILTROS DE NEGOCIO (Calidad de Datos)
# Filtro de NIA según tus especificaciones (Rango 2MM - 3MM)
df_silver = df_silver.filter(
    (F.col("logs_nia") >= 2000000000) & 
    (F.col("logs_nia") <= 2999999999)
)

# 5. NORMALIZACIÓN DE TEXTO Y FECHAS
mapa_ubicaciones = {"Calificacion": "Calificación", "Iman Core": "Imán"}

# Aplicamos el mapeo de nombres y convertimos a zona horaria Lima
df_silver = df_silver.withColumn(
    "logs_ubicacion", 
    F.coalesce(F.create_map([F.lit(x) for x in [val for pair in mapa_ubicaciones.items() for val in pair]]).getItem(F.col("logs_ubicacion")), F.col("logs_ubicacion"))
).withColumn(
    "evento_fecha_lima", 
    F.from_utc_timestamp(F.from_unixtime(F.col("evento_ts")/1000), "America/Lima")
)

# 6. ELIMINACIÓN DE DUPLICADOS
# Garantizamos que no haya eventos repetidos por NIA y Tiempo
df_silver = df_silver.dropDuplicates(["logs_nia", "evento_ts"])

# 7. GUARDAR EN TABLA DELTA SILVER
TABLA_SILVER = "telemetria_patio_silver"
df_silver.write.format("delta").mode("overwrite").saveAsTable(TABLA_SILVER)

print(f"✨ Capa Silver finalizada. Datos limpios en: {TABLA_SILVER}")
display(df_silver.limit(10))